# 12 · Generalisation beyond a random split: unseen hosts and time

A random test split is a sample of the same capture: near-duplicate flows (same host, same time window, same attack run) can sit in both training and test data.
This notebook draws **one** uniform random sample of rows and builds every split from it in memory. The same model configuration is trained on the same number of rows under each protocol:

| Protocol | How training and test rows are separated |
|---|---|
| **Random split** (reference) | stratified random 80 / 20 |
| **Unseen-host split**, repeated | all flows of a hashed source-IP bucket go to one partition (60 / 20 / 20 of the buckets); the assignment is redrawn `GEN_HOST_REPEATS` times (mean ± std) |
| **Time split** (datasets with a timestamp) | train on the first 60 % of the capture, test on the last 20 % |

**Diagnostics**
* **Hosts per class**: how many distinct source-IP buckets each class has. A class with 1–3 hosts ends up entirely in training or entirely in test.
* **Host-fingerprint ablation**: the same protocols without `MIN_TTL, MAX_TTL, TCP_WIN_MAX_IN, TCP_WIN_MAX_OUT` (operating-system fingerprints). Set `GEN_FINGERPRINT_COLS` to ablate another group (e.g. only `MIN_TTL,MAX_TTL`) and `GEN_OUT_SUFFIX` (e.g. `_ttl_only`) so the results go to their own folder.

No label, IP or time information is used as a feature; they only decide which rows go where.
Labels in the summary: drop ≤ 0.01 MCC = robust · 0.01–0.05 = moderate · > 0.05 = substantial.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
import feature_policy as fp, generalization as gz
import time
print("Dataset:", ACTIVE_DATASET, "| sampled rows:", f"{GEN_SAMPLE_ROWS:,}", "| train rows per protocol:", f"{GEN_TRAIN_ROWS:,}", "| test cap:", f"{GEN_EVAL_ROWS:,}", "| host repeats:", GEN_HOST_REPEATS)
assert Path(DEFAULT_DATASET_PATH).exists(), f"CSV not found: {DEFAULT_DATASET_PATH}"

## 1 · Columns, labels, model configuration

In [ ]:
header = pd.read_csv(DEFAULT_DATASET_PATH, nrows=0).columns.tolist()
groups = fp.classify_columns(header, TARGET_COL)
core = fp.numeric_columns(DEFAULT_DATASET_PATH, groups["core"])
le, y_all = fp.read_target(DEFAULT_DATASET_PATH, TARGET_COL); class_names = list(le.classes_); n_total = len(y_all)
base = gz.base_name(ACTIVE_DATASET); src_col, time_col = gz.SRC_IP_COL.get(base), gz.TIME_COL.get(base)
try:
    artifact, metadata = load_artifact_and_metadata(); BASE_PARAMS, BASE_K = dict(metadata["best_params"]), int(metadata["best_k"])
except FileNotFoundError:
    BASE_PARAMS, BASE_K = {"max_depth": 20, "min_samples_leaf": 8}, 30
fing = [c for c in GEN_FINGERPRINT_COLS if c in core]
core_nofp = [c for c in core if c not in fing]
print(f"{n_total:,} rows | {len(core)} features | {len(class_names)} classes | source-IP column: {src_col} | time column: {time_col}")
print("model:", BASE_PARAMS, "| K =", min(BASE_K, len(core)), "| host-fingerprint columns present:", fing or "none (ablation skipped)")

## 2 · ONE uniform random sample of rows, with host bucket and time per row

In [ ]:
t0 = time.perf_counter()
S = min(GEN_SAMPLE_ROWS, n_total)
idx = np.sort(np.random.default_rng(SEED).choice(n_total, S, replace=False))                  # uniform random rows (NOT a file prefix)
XS = fp.read_rows(DEFAULT_DATASET_PATH, core, {"pool": idx, "val": np.array([], dtype=np.int64), "test": np.array([], dtype=np.int64)})["pool"]
assert (XS.index.values == idx).all(), "row alignment problem"
yS = y_all[idx]
HAS_HOST = bool(src_col and src_col in header); HAS_TIME = bool(time_col and time_col in header)
bS = gz.read_hash_buckets(DEFAULT_DATASET_PATH, src_col)[idx] if HAS_HOST else None
tS = gz.read_time_values(DEFAULT_DATASET_PATH, time_col)[idx] if HAS_TIME else None
XS = XS.reset_index(drop=True)
print(f"sample {len(XS):,} rows x {XS.shape[1]} features loaded in {time.perf_counter() - t0:.0f}s | class shares in the sample:",
      {class_names[i]: f"{(yS == i).mean():.3%}" for i in np.argsort(-np.bincount(yS, minlength=len(class_names)))[:len(class_names)]})

## 3 · Diagnostics: how many hosts does each class have?

In [ ]:
SUFFIX = os.environ.get("GEN_OUT_SUFFIX", "")          # e.g. "_ttl_only" when GEN_FINGERPRINT_COLS is changed
OUT = Path("generalization") / (ACTIVE_DATASET + SUFFIX); OUT.mkdir(parents=True, exist_ok=True)
if HAS_HOST:
    trm, vam, tem = gz.bucket_masks(bS, SEED)
    hosts = gz.class_host_table(yS, bS, class_names, trm, tem).sort_values("distinct_host_buckets")
    hosts["single_partition"] = (hosts.buckets_in_train == 0) | (hosts.buckets_in_test == 0)
    hosts.to_csv(OUT / "generalization_hosts_per_class.csv", index=False); display(hosts.round(3))
    few = hosts[hosts.distinct_host_buckets <= 10]
    if len(few): print("classes with <= 10 host buckets (a host split cannot test these properly):", ", ".join(f"{r['class']} ({r.distinct_host_buckets})" for _, r in few.iterrows()))
else:
    print("no source-IP column -> host diagnostics skipped")

## 4 · Train the same configuration under every protocol

In [ ]:
def run(protocol, tr_idx, te_idx, cols, fset, rep):
    tr = gz.safe_sample(tr_idx, yS[tr_idx], GEN_TRAIN_ROWS, SEED + rep); te = gz.safe_sample(te_idx, yS[te_idx], GEN_EVAL_ROWS, SEED + rep)
    if len(tr) == 0 or len(te) == 0: return None
    miss = gz.classes_missing_from_train(yS[tr], yS[te]); rare, xrare = infer_rare_classes(yS[tr])
    pipe = build_safe_pipeline(DecisionTreeClassifier(random_state=SEED, **BASE_PARAMS), k=min(BASE_K, len(cols))); pipe.fit(XS.iloc[tr][cols], yS[tr])
    pr = pipe.predict(XS.iloc[te][cols]); m = compute_enhanced_metrics(yS[te], pr, rare, xrare)
    cmx = confusion_matrix(yS[te], pr, labels=np.arange(len(class_names))); rc = cmx.diagonal() / np.maximum(cmx.sum(axis=1), 1)
    row = {"protocol": protocol, "features": fset, "repeat": rep, "train_rows": len(tr), "test_rows": len(te), "mcc": m["mcc"], "macro_f1": m["macro_f1"], "accuracy": m["accuracy"],
           "nodes": int(pipe.named_steps["clf"].tree_.node_count), "test_classes_absent_from_train": json.dumps({class_names[k]: v for k, v in miss.items()}), "rows_of_absent_classes": int(sum(miss.values()))}
    pcs = [{"protocol": protocol, "features": fset, "repeat": rep, "class": c, "support": int(cmx[i].sum()), "recall": float(rc[i]) if cmx[i].sum() else np.nan} for i, c in enumerate(class_names)]
    return row, pcs

FSETS = {"all features": core}
if fing: FSETS["without host-fingerprint features"] = core_nofp
runs, pcs_all = [], []
t0 = time.perf_counter()
for fset, cols in FSETS.items():
    for rep in range(GEN_HOST_REPEATS):                                               # random split, several seeds -> same repeats as the host split
        a, b = train_test_split(np.arange(len(yS)), test_size=0.2, stratify=yS, random_state=SEED + rep)
        out = run("random split (reference)", a, b, cols, fset, rep)
        if out: runs.append(out[0]); pcs_all += out[1]
        if HAS_HOST:
            trm, vam, tem = gz.bucket_masks(bS, SEED + rep)
            out = run("unseen-host split", np.flatnonzero(trm), np.flatnonzero(tem), cols, fset, rep)
            if out: runs.append(out[0]); pcs_all += out[1]
    if HAS_TIME:
        ok = ~np.isnan(tS); q60, q80 = np.nanquantile(tS, [.6, .8])
        out = run("time split", np.flatnonzero(ok & (tS <= q60)), np.flatnonzero(ok & (tS > q80)), cols, fset, 0)
        if out: runs.append(out[0]); pcs_all += out[1]
    print(f"{fset:36s} done ({time.perf_counter() - t0:.0f}s)", flush=True)
R = pd.DataFrame(runs); R.to_csv(OUT / "generalization_runs.csv", index=False); pd.DataFrame(pcs_all).to_csv(OUT / "generalization_per_class.csv", index=False)

## 5 · Summary

In [ ]:
g = R.groupby(["features", "protocol"]).agg(mcc=("mcc", "mean"), mcc_std=("mcc", "std"), macro_f1=("macro_f1", "mean"), accuracy=("accuracy", "mean"), n_runs=("mcc", "size"),
                                            train_rows=("train_rows", "mean"), test_rows=("test_rows", "mean"), rows_of_absent_classes=("rows_of_absent_classes", "mean")).reset_index()
ref = g[g.protocol.str.startswith("random")].set_index("features")["mcc"]
g["mcc_drop_vs_random"] = g.apply(lambda r: ref[r["features"]] - r["mcc"], axis=1)
g["verdict"] = np.where(g.protocol.str.startswith("random"), "reference", np.where(g.mcc_drop_vs_random <= 0.01, "robust", np.where(g.mcc_drop_vs_random <= 0.05, "moderate drop", "substantial drop")))
# summary file: the 'all features' rows
main = g[g.features == "all features"].drop(columns=["features"]); main.to_csv(OUT / "generalization_splits.csv", index=False); g.to_csv(OUT / "generalization_summary_all.csv", index=False)
display(g.round(4))
print("\nsanity check: the random-split MCC should be close to the main model trained on this many rows (about 0.965 at 1M training rows for NF-ToN-IoT-v2); a much lower value means a protocol problem.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
protos = list(dict.fromkeys(g.protocol)); fs = list(dict.fromkeys(g.features)); w = .8 / len(fs); colr = ["#1f77b4", "#d95f02", "#2ca02c"]
ax = axes[0]
for j, f in enumerate(fs):
    d = g[g.features == f].set_index("protocol").reindex(protos)
    ax.bar(np.arange(len(protos)) + j * w, d["mcc"], w, yerr=d["mcc_std"].fillna(0), capsize=3, color=colr[j], label=f)
    for i, (v, p_) in enumerate(zip(d["mcc"], protos)): ax.text(i + j * w, v + .01, f"{v:.3f}", ha="center", fontsize=7)
ax.set_xticks(np.arange(len(protos)) + w * (len(fs) - 1) / 2, [p.replace(" (reference)", "") for p in protos]); ax.set_ylim(0, 1.08); ax.set_ylabel("test MCC (mean ± std over repeats)"); ax.legend(fontsize=7)
ax.set_title(f"{ACTIVE_DATASET}: same model, different evaluation protocol")
pc = pd.DataFrame(pcs_all); pc = pc[pc.features == "all features"]
piv = pc.groupby(["class", "protocol"])["recall"].mean().unstack("protocol").reindex(class_names)[[p for p in protos if p in set(pc.protocol)]]
im = axes[1].imshow(piv.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto"); axes[1].grid(False)
axes[1].set_xticks(range(piv.shape[1]), [c.split(" ")[0] for c in piv.columns]); axes[1].set_yticks(range(len(piv)), piv.index, fontsize=7)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        if not np.isnan(piv.values[i, j]): axes[1].text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
axes[1].set_title("per-class recall (all features, mean over repeats)"); fig.colorbar(im, ax=axes[1], shrink=.8)
fig.savefig(FIGURES_DIR / f"generalization_splits{SUFFIX}.png", dpi=300, bbox_inches="tight"); plt.show()

## 6 · Per-protocol summary

In [ ]:
a = g[g.features == "all features"].set_index("protocol")
print(f"random split (reference): MCC {a.loc[[p for p in a.index if p.startswith('random')][0], 'mcc']:.4f}")
for p in a.index:
    if p.startswith("random"): continue
    r = a.loc[p]; print(f"{p:22s}: MCC {r.mcc:.4f} +- {0 if np.isnan(r.mcc_std) else r.mcc_std:.4f}  (drop {r.mcc_drop_vs_random:+.4f}) -> {r.verdict.upper()}")
if len(fs) > 1 and "unseen-host split" in set(g.protocol):
    h = g[g.protocol == "unseen-host split"].set_index("features")["mcc"]; rr = g[g.protocol.str.startswith("random")].set_index("features")["mcc"]
    print(f"\nhost-fingerprint ablation: random split {rr.iloc[0]:.4f} -> {rr.iloc[1]:.4f} ; unseen-host split {h.iloc[0]:.4f} -> {h.iloc[1]:.4f}")
    if h.iloc[1] - h.iloc[0] > 0.05 and rr.iloc[0] - rr.iloc[1] < 0.02:
        print("  -> removing OS-fingerprint features barely changes the random split but helps unseen hosts: the model was leaning on machine fingerprints (TTL / TCP window).")
    elif rr.iloc[0] - rr.iloc[1] > 0.05:
        print("  -> the RANDOM-split score itself depends on these features: they carry host identity (TTL / TCP window fingerprint a machine), so a random split lets the model recognise machines it has seen.")
        print("     Treat the random-split MCC as an in-capture number; report the host-split number next to it.")
    elif h.iloc[1] - h.iloc[0] < 0.02: print("  -> removing those features does not rescue the host split: the dependence on hosts is broader than OS fingerprints.")
if HAS_HOST:
    one = hosts[hosts.single_partition]
    if len(one): print("\nclasses that fall entirely into train OR test under the host split (recall there is 0 by construction, not a model failure):", ", ".join(one["class"]))
print("\nA drop is NOT a bug: it measures how much a random split flatters the model. Report it next to the headline number.")